# Diffusion Microscope — Experiment Runner

Runs all three experiments against GPT-2 and Pythia-410m and saves results to Google Drive.

**Runtime:** Select `Runtime → Change runtime type → T4 GPU` before running.

---
**Experiments:**
- **Exp 1 — Alpha compression visibility:** does alpha=1 produce more distinct images than alpha=1000? (both models, last layer)
- **Exp 2 — L0→L1 bottleneck:** what does the first transformer layer discard? (Pythia only, alpha=1, layers 0–3)
- **Exp 3 — Alpha sensitivity as novelty detector:** does LPIPS(alpha=1, alpha=1000) rank unusual prompts above common ones?

**Session persistence:** HuggingFace model cache and experiment results are stored on Drive — re-running skips completed work.

## 1 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — switch runtime to T4 GPU before continuing.')

In [ ]:
import os

# ── Edit these paths if needed ──────────────────────────────────────────────
REPO_URL    = 'https://github.com/leonorae/slicer'   # replace with your fork if needed
REPO_BRANCH = 'claude/geometric-visualization-pipeline-kHzEj'
REPO_DIR    = '/content/slicer'
DRIVE_BASE  = '/content/drive/MyDrive/diffusion_microscope'
# ────────────────────────────────────────────────────────────────────────────

HF_CACHE_DIR    = os.path.join(DRIVE_BASE, 'hf_cache')
RESULTS_GPT2    = os.path.join(DRIVE_BASE, 'experiment_results_nb_gpt2')
RESULTS_PYTHIA  = os.path.join(DRIVE_BASE, 'experiment_results_nb_pythia')

os.makedirs(HF_CACHE_DIR,   exist_ok=True)
os.makedirs(RESULTS_GPT2,   exist_ok=True)
os.makedirs(RESULTS_PYTHIA, exist_ok=True)

# Point HuggingFace at the Drive cache so large models survive session resets
os.environ['HF_HOME']             = HF_CACHE_DIR
os.environ['TRANSFORMERS_CACHE']  = HF_CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR

print('Drive base :', DRIVE_BASE)
print('HF cache   :', HF_CACHE_DIR)
print('GPT-2 out  :', RESULTS_GPT2)
print('Pythia out :', RESULTS_PYTHIA)

## 2 · Clone repo and install

In [ ]:
import os

if os.path.isdir(REPO_DIR):
    print('Repo already cloned — pulling latest.')
    !git -C {REPO_DIR} fetch origin {REPO_BRANCH}
    !git -C {REPO_DIR} checkout {REPO_BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}

In [ ]:
# Colab already ships torch+CUDA — install everything else and avoid overwriting torch.
!pip install -q \
    open-clip-torch \
    diffusers \
    Pillow \
    lpips \
    datasets \
    nltk \
    sentencepiece \
    accelerate \
    scikit-learn \
    umap-learn

# Install the package itself (no deps — already installed above)
!pip install -q -e {REPO_DIR} --no-deps

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('Install complete.')

## 3 · Patch configs with Drive output paths

The repo ships configs pointing to local `./experiment_results_nb_*` dirs.
This cell rewrites the output paths to point at Drive so results persist.

In [ ]:
import json, shutil

def patch_config(src_name, out_dir):
    src = os.path.join(REPO_DIR, src_name)
    with open(src) as f:
        cfg = json.load(f)
    cfg['output']['base_dir'] = out_dir
    dst = os.path.join(REPO_DIR, src_name)  # overwrite in-place for CLI use
    with open(dst, 'w') as f:
        json.dump(cfg, f, indent=2)
    print(f'{src_name} → {out_dir}')

patch_config('experiment_config_nb_gpt2.json',   RESULTS_GPT2)
patch_config('experiment_config_nb_pythia.json',  RESULTS_PYTHIA)

## 4 · GPT-2 experiments

Covers **Exp 1** (alpha compression) and **Exp 3** (novelty detector).

Phases: `train` → `generate` → `grids`

All phases are idempotent — if interrupted, re-run the cell and it will pick up from where it left off.

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_gpt2.json \
    --phase train

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_gpt2.json \
    --phase generate

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_gpt2.json \
    --phase grids

## 5 · Pythia-410m experiments

Covers **Exp 1** (alpha compression, L23), **Exp 2** (L0→L1 bottleneck, L0-3), and **Exp 3** (novelty detector, L23).

Pythia-410m is ~800 MB. First run downloads it to the Drive HF cache.

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_pythia.json \
    --phase train

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_pythia.json \
    --phase generate

In [ ]:
%%time
!cd {REPO_DIR} && python run_experiment.py \
    --config experiment_config_nb_pythia.json \
    --phase grids

## 6 · Results

### 6a · Exp 1 — Alpha compression: alpha=1 vs alpha=1000 side-by-side

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def find_images(results_dir, layer, alpha, seed=42):
    """Return {probe_slug: image_path} for a given layer/alpha/seed."""
    pattern = os.path.join(
        results_dir,
        'grids', 'by_projection',
        f'per_layer_alpha{alpha}',
        '*',  # probe slug
        'per_layer',
        f'L{layer}_CFG7.5_seed{seed}.png'
    )
    paths = glob.glob(pattern)
    return {os.path.basename(os.path.dirname(os.path.dirname(p))): p for p in sorted(paths)}

def show_alpha_comparison(results_dir, layer, probes_of_interest, seed=42, title=''):
    imgs_low  = find_images(results_dir, layer, 1,    seed)
    imgs_high = find_images(results_dir, layer, 1000, seed)

    common = [p for p in probes_of_interest if p in imgs_low and p in imgs_high]
    if not common:
        print(f'No matching images found in {results_dir}')
        print('Available probe slugs:', list(imgs_low.keys())[:5], '...')
        return

    fig, axes = plt.subplots(2, len(common), figsize=(3 * len(common), 7))
    fig.suptitle(title or f'Layer {layer} — alpha=1 (top) vs alpha=1000 (bottom)', fontsize=12)

    for col, slug in enumerate(common):
        for row, (imgs, label) in enumerate([(imgs_low, 'α=1'), (imgs_high, 'α=1000')]):
            ax = axes[row][col] if len(common) > 1 else axes[row]
            ax.imshow(Image.open(imgs[slug]))
            ax.axis('off')
            if row == 0:
                ax.set_title(slug.replace('_', ' '), fontsize=8)
            if col == 0:
                ax.set_ylabel(label, fontsize=9)

    plt.tight_layout()
    plt.show()

# GPT-2 — Exp 1 probes
exp1_probes = ['a_cat', 'a_dog', 'democracy', 'entropy']
show_alpha_comparison(RESULTS_GPT2, layer=11, probes_of_interest=exp1_probes,
                      title='GPT-2 L11 — concrete vs abstract at alpha=1 vs 1000')

In [ ]:
# Pythia — Exp 1 probes
show_alpha_comparison(RESULTS_PYTHIA, layer=23, probes_of_interest=exp1_probes,
                      title='Pythia-410m L23 — concrete vs abstract at alpha=1 vs 1000')

### 6b · Exp 2 — L0→L1 bottleneck in Pythia (alpha=1, layers 0–3)

In [ ]:
def show_layer_sweep(results_dir, alpha, layers, probe_slug, seed=42, title=''):
    """One row per layer for a single probe."""
    paths = []
    for L in layers:
        pattern = os.path.join(
            results_dir, 'grids', 'by_projection',
            f'per_layer_alpha{alpha}', probe_slug, 'per_layer',
            f'L{L}_CFG7.5_seed{seed}.png'
        )
        matches = glob.glob(pattern)
        paths.append((L, matches[0] if matches else None))

    valid = [(L, p) for L, p in paths if p]
    if not valid:
        print(f'No images found for probe "{probe_slug}" — check slug spelling.')
        return

    fig, axes = plt.subplots(1, len(valid), figsize=(3 * len(valid), 4))
    fig.suptitle(title or f'{probe_slug} — layer sweep at alpha={alpha}', fontsize=11)

    for i, (L, p) in enumerate(valid):
        ax = axes[i] if len(valid) > 1 else axes
        ax.imshow(Image.open(p))
        ax.axis('off')
        ax.set_title(f'L{L}', fontsize=10)

    plt.tight_layout()
    plt.show()

# Show L0→L3 sweep at alpha=1 for a concrete and abstract probe
for probe in ['a_cat', 'democracy']:
    show_layer_sweep(RESULTS_PYTHIA, alpha=1, layers=[0, 1, 2, 3], probe_slug=probe,
                     title=f'Pythia L0→L3 at alpha=1 — "{probe.replace("_", " ")}": what does L1 discard?')

### 6c · Exp 3 — Alpha sensitivity as novelty detector

In [ ]:
# Three tiers: common-concrete, common-abstract, unusual
# LPIPS is computed by the metrics phase; here we show the visual comparison.

tiers = {
    'common-concrete': ['a_cat', 'a_dog', 'a_house'],
    'common-abstract': ['democracy', 'justice', 'beauty'],
    'unusual':         [
        'the_feeling_of_almost_remembering',
        'the_color_of_tuesday',
        'entropy_at_midnight'
    ]
}

for model_name, results_dir, last_layer in [
    ('GPT-2',       RESULTS_GPT2,   11),
    ('Pythia-410m', RESULTS_PYTHIA, 23),
]:
    print(f'\n=== {model_name} — novelty tier comparison ===')
    for tier_name, probes in tiers.items():
        show_alpha_comparison(
            results_dir, layer=last_layer,
            probes_of_interest=probes,
            title=f'{model_name} L{last_layer} — {tier_name} — alpha=1 (top) vs 1000 (bottom)'
        )

### 6d · LPIPS summary — alpha sensitivity by tier

Runs the `metrics` phase to compute LPIPS, then plots mean LPIPS per tier.

In [ ]:
%%time
for config in ['experiment_config_nb_gpt2.json', 'experiment_config_nb_pythia.json']:
    print(f'\n--- metrics: {config} ---')
    !cd {REPO_DIR} && python run_experiment.py --config {config} --phase metrics

In [ ]:
import json
import numpy as np

tier_map = {
    'a cat': 'common-concrete', 'a dog': 'common-concrete', 'a house': 'common-concrete',
    'democracy': 'common-abstract', 'justice': 'common-abstract', 'beauty': 'common-abstract',
    'the feeling of almost remembering': 'unusual',
    'the color of Tuesday': 'unusual',
    'entropy at midnight': 'unusual',
}

def plot_lpips_by_tier(results_dir, model_name, last_layer, alpha_pair=(1, 1000)):
    manifest_path = os.path.join(results_dir, 'manifest.json')
    if not os.path.exists(manifest_path):
        print(f'No manifest found at {manifest_path}')
        return

    with open(manifest_path) as f:
        manifest = json.load(f)

    metrics = manifest.get('metrics', {})
    if not metrics:
        print(f'No metrics in manifest for {model_name} — run metrics phase first.')
        return

    # Collect LPIPS values keyed by probe text and alpha
    tier_lpips = {'common-concrete': [], 'common-abstract': [], 'unusual': []}

    for key, val in metrics.items():
        # key format varies by implementation — inspect a few
        pass

    print(f'{model_name} manifest keys (sample):', list(manifest.keys())[:10])
    print('Metrics keys (sample):', list(metrics.keys())[:5] if metrics else 'empty')

for model_name, results_dir, last_layer in [
    ('GPT-2',       RESULTS_GPT2,   11),
    ('Pythia-410m', RESULTS_PYTHIA, 23),
]:
    plot_lpips_by_tier(results_dir, model_name, last_layer)

### 6e · Full grid images

Display the composed grid PNGs (all probes × seeds at a given alpha).

In [ ]:
from IPython.display import display

def show_grid_images(results_dir, alpha, seed=42):
    pattern = os.path.join(
        results_dir, 'grids', 'by_projection',
        f'per_layer_alpha{alpha}', '*',
        f'grid_seed{seed}.png'
    )
    grid_paths = sorted(glob.glob(pattern))
    for p in grid_paths:
        probe_slug = os.path.basename(os.path.dirname(p))
        print(f'\n{probe_slug} (alpha={alpha})')
        display(Image.open(p))

for alpha in [1, 1000]:
    print(f'\n========== GPT-2 alpha={alpha} ==========')
    show_grid_images(RESULTS_GPT2, alpha)

for alpha in [1, 1000]:
    print(f'\n========== Pythia-410m alpha={alpha} ==========')
    show_grid_images(RESULTS_PYTHIA, alpha)

## 7 · Save summary

Results are already on Drive (output dirs point there). This cell writes a short summary of what ran.

In [ ]:
import datetime

summary_path = os.path.join(DRIVE_BASE, 'run_summary.txt')
lines = [
    f'Run completed: {datetime.datetime.now().isoformat()}',
    f'Branch: {REPO_BRANCH}',
    '',
    'GPT-2 results:',
]
for root, dirs, files in os.walk(RESULTS_GPT2):
    png_count = sum(1 for f in files if f.endswith('.png'))
    if png_count:
        lines.append(f'  {os.path.relpath(root, RESULTS_GPT2)}: {png_count} images')

lines.append('')
lines.append('Pythia-410m results:')
for root, dirs, files in os.walk(RESULTS_PYTHIA):
    png_count = sum(1 for f in files if f.endswith('.png'))
    if png_count:
        lines.append(f'  {os.path.relpath(root, RESULTS_PYTHIA)}: {png_count} images')

summary = '\n'.join(lines)
print(summary)
with open(summary_path, 'w') as f:
    f.write(summary)
print(f'\nSummary written to {summary_path}')